# 2. Preprocesamiento y Preparación de Datos

En este notebook, prepararemos los datos para el entrenamiento de los modelos de Machine Learning. Los pasos incluyen:
1.  Cargar el dataset.
2.  Aplicar One-Hot Encoding a las variables categóricas.
3.  Escalar las variables numéricas.
4.  Dividir el dataset en conjuntos de entrenamiento, validación y prueba.
5.  Exportar los conjuntos de datos procesados.

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import joblib

# Definir rutas
data_path = '../data/cars_hyundai.csv'
processed_path = '../data/processed'

# Crear el directorio de datos procesados si no existe
if not os.path.exists(processed_path):
    os.makedirs(processed_path)

## Cargar y Preparar los Datos

In [2]:
# Cargar el dataset
try:
    df = pd.read_csv(data_path)
    df.columns = ['engine_temp', 'brake_pad_thickness', 'tire_pressure', 'maintenance_type', 'anomaly']
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'cars_hyundai.csv'. Ejecuta el notebook 01 primero.")
    df = None

if df is not None:
    # Separar características (X) y variable objetivo (y)
    X = df.drop('anomaly', axis=1)
    y = df['anomaly']
    
    # Identificar columnas numéricas y categóricas
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
    
    print("\nCaracterísticas numéricas:", numerical_features)
    print("Características categóricas:", categorical_features)

Dataset cargado exitosamente.

Características numéricas: ['engine_temp', 'brake_pad_thickness', 'tire_pressure']
Características categóricas: ['maintenance_type']


### One-Hot Encoding para `maintenance_type`

In [3]:
if df is not None:
    # Aplicar One-Hot Encoding
    X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)
    
    print("Forma de X después del One-Hot Encoding:", X_encoded.shape)
    print("Nuevas columnas:", X_encoded.columns.tolist())
    display(X_encoded.head())

Forma de X después del One-Hot Encoding: (1100, 5)
Nuevas columnas: ['engine_temp', 'brake_pad_thickness', 'tire_pressure', 'maintenance_type_Repair', 'maintenance_type_Routine Maintenance']


,engine_temp,brake_pad_thickness,tire_pressure,maintenance_type_Repair,maintenance_type_Routine Maintenance
0,81.022390,7.984018,35.964546,True,False
1,98.076029,10.718692,32.143593,False,True
2,81.205967,10.983070,31.058628,False,True
3,86.081294,7.045311,28.539264,True,False
4,93.496568,9.948991,33.599560,False,False


### Escalado de Variables Numéricas

Usamos `StandardScaler` para escalar las características numéricas, de modo que tengan media 0 y desviación estándar 1. Esto es importante para muchos algoritmos de ML.

In [4]:
if 'X_encoded' in locals():
    # Guardar los nombres de las columnas antes de escalar
    feature_names = X_encoded.columns.tolist()
    
    # Inicializar el escalador
    scaler = StandardScaler()
    
    # Escalar los datos
    X_scaled = scaler.fit_transform(X_encoded)
    
    # Convertir de nuevo a DataFrame para visualización
    X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)
    
    print("Estadísticas descriptivas después del escalado:")
    display(X_scaled_df.describe().T)
    
    # Guardar el escalador para usarlo en producción/inferencia
    scaler_path = os.path.join('../models', 'scaler.pkl')
    joblib.dump(scaler, scaler_path)
    print(f"\nEscalador guardado en: {scaler_path}")
    
    # Guardar los nombres de las características
    features_path = os.path.join('../models', 'feature_names.pkl')
    joblib.dump(feature_names, features_path)
    print(f"Nombres de características guardados en: {features_path}")

Estadísticas descriptivas después del escalado:


,count,mean,std,min,25%,50%,75%,max
engine_temp,1100.0,-2.144547e-15,1.000455,-1.739796,-0.904186,0.061007,0.828041,1.703578
brake_pad_thickness,1100.0,7.977457e-16,1.000455,-1.718489,-0.902540,0.030651,0.827155,1.731412
tire_pressure,1100.0,-1.195004e-15,1.000455,-1.767066,-0.841646,0.027332,0.881576,1.704659
maintenance_type_Repair,1100.0,2.906766e-17,1.000455,-0.704698,-0.704698,-0.704698,1.419048,1.419048
maintenance_type_Routine Maintenance,1100.0,4.521636e-17,1.000455,-0.704698,-0.704698,-0.704698,1.419048,1.419048



Escalador guardado en: ../models/scaler.pkl
Nombres de características guardados en: ../models/feature_names.pkl


## Partición Estratificada de Datos

Dividimos el dataset en tres conjuntos:
*   **Entrenamiento (70%)**: Para entrenar el modelo.
*   **Validación (15%)**: Para ajustar hiperparámetros.
*   **Prueba (15%)**: Para la evaluación final del modelo.

Usamos una partición estratificada para mantener la misma proporción de clases en cada conjunto.

In [5]:
if 'X_scaled' in locals():
    # Primera división: 70% entrenamiento, 30% temporal (para validación y prueba)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_scaled, y, 
        test_size=0.30, 
        random_state=42, 
        stratify=y
    )

    # Segunda división: 50% del temporal para validación, 50% para prueba (15% del total cada uno)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

    print(f"Forma de X_train: {X_train.shape}")
    print(f"Forma de y_train: {y_train.shape}")
    print(f"Forma de X_val:   {X_val.shape}")
    print(f"Forma de y_val:   {y_val.shape}")
    print(f"Forma de X_test:  {X_test.shape}")
    print(f"Forma de y_test:  {y_test.shape}")

Forma de X_train: (770, 5)
Forma de y_train: (770,)
Forma de X_val:   (165, 5)
Forma de y_val:   (165,)
Forma de X_test:  (165, 5)
Forma de y_test:  (165,)


### Verificar Distribución de Clases

In [6]:
if 'y_train' in locals():
    print("Distribución de clases en el conjunto original:")
    print(y.value_counts(normalize=True))
    print("\nDistribución de clases en el conjunto de entrenamiento:")
    print(y_train.value_counts(normalize=True))
    print("\nDistribución de clases en el conjunto de validación:")
    print(y_val.value_counts(normalize=True))
    print("\nDistribución de clases en el conjunto de prueba:")
    print(y_test.value_counts(normalize=True))

Distribución de clases en el conjunto original:
anomaly
1    0.500909
0    0.499091
Name: proportion, dtype: float64

Distribución de clases en el conjunto de entrenamiento:
anomaly
1    0.501299
0    0.498701
Name: proportion, dtype: float64

Distribución de clases en el conjunto de validación:
anomaly
1    0.50303
0    0.49697
Name: proportion, dtype: float64

Distribución de clases en el conjunto de prueba:
anomaly
0    0.50303
1    0.49697
Name: proportion, dtype: float64


## Exportar los Datos Procesados

Guardamos los conjuntos de datos como archivos `.npy` para usarlos en los siguientes notebooks.

In [7]:
if 'X_train' in locals():
    # Guardar los arrays de numpy
    np.save(os.path.join(processed_path, 'X_train.npy'), X_train)
    np.save(os.path.join(processed_path, 'y_train.npy'), y_train)
    np.save(os.path.join(processed_path, 'X_val.npy'), X_val)
    np.save(os.path.join(processed_path, 'y_val.npy'), y_val)
    np.save(os.path.join(processed_path, 'X_test.npy'), X_test)
    np.save(os.path.join(processed_path, 'y_test.npy'), y_test)
    
    print(f"Datos procesados guardados en la carpeta: {processed_path}")
    # Listar archivos para confirmar
    print("\nArchivos en el directorio 'processed':")
    print(os.listdir(processed_path))

Datos procesados guardados en la carpeta: ../data/processed

Archivos en el directorio 'processed':
['y_train.npy', 'y_test.npy', 'X_test.npy', 'y_val.npy', 'X_train.npy', 'X_val.npy']
